# Figures and tables with `anb.analysis`

The paper's figures and tables from **whatever runs a simulation folder holds**: one figure at a time, then all of
them with `make_all`.

- Every figure function finds its runs in the folder by parameter, uses every seed it finds and returns a
  `FigureResult` (`figure`, the plotted numbers as `data`, the file stem `name`) without saving anything.
- A figure whose runs are missing warns with `FigureNotApplicable`, naming them, and returns `None`. The cell then
  shows only the warning. Fewer seeds than the paper used give a `FewSeedsWarning`, and the figure is still drawn.
- Table functions return the LaTeX, and summaries return DataFrames.

What each figure needs, its keyword arguments (with the paper's choices as defaults) and the file names are in
`docu/analysis-package.md`. The old `analyse_*.py` scripts stay the reference for the paper's figures
(`docu/analysis.md`).

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from anb.analysis import (Baseline, ExampleAgentFallback, FewSeedsWarning, FigureNotApplicable, FigureResult,
                          FitWarning, Loader, MissingLevelsWarning, example_agent_stats, fig2, fig3, fig4, fig5, fig6,
                          figA3, figA4, figA7, figA8, focal_sign_counts, greedy_uncorrelated_subset, index_folder,
                          make_all, polarisation_summary, save_table, table3, tableA5, tableA6, tableA7)

# Show the package's warnings every time, also when a cell runs again (Python shows a repeated warning only once).
for category in (FigureNotApplicable, FewSeedsWarning, MissingLevelsWarning, FitWarning, ExampleAgentFallback):
    warnings.simplefilter("always", category)

## Setup

The paths are relative to `notebooks/`. `BASE` is the base config, the non-swept parameters every run of a figure
must have. `Baseline()` is the paper's adaptive baseline. For a variant, override its fields, e.g.
`Baseline(link_prob=0.2)` or `Baseline(M=15)`: the figures then use the variant's runs, and their names get the
differing fields appended (`fig3_responseFrequencies_link_prob0.20`), so they don't overwrite the paper's.

In [ ]:
SIM_DIR = Path("../simOut_old_model")  # the simulation folder: ensemble CSVs, detailed runs in its detailed/
OUT_DIR = Path("../figs")              # where figures are saved (tables into its tables/), created if missing
BASE = Baseline()                      # the paper's adaptive baseline
FORMATS = ("png",)                     # add "pdf" for the paper

## The folder

`index_folder` parses every CSV name in `SIM_DIR` and `SIM_DIR/detailed/` into one row of parameters. Files that
don't follow the naming scheme are left out, with one warning. Index once and pass the index to every function
(`index=`), and share one `Loader` (`loader=`), so each CSV is read from disk once. `Loader(whole=True)` reads every
column the figures use on the first request, as `make_all` does. On `simOut_old_model` it holds about 1.9 GB once
everything has run, and `Loader()` holds about half of that but reads some CSVs twice.

The table counts the seeds: one row per combination of the parameters that vary in the folder (and ensemble or
detailed), one column per pressure `s`.

In [ ]:
index = index_folder(SIM_DIR)
loader = Loader(whole=True)
common = dict(base=BASE, loader=loader, index=index)

varying = [c for c in index.columns if c not in ("s", "seed", "path", "detailed") and index[c].nunique() > 1]
index.groupby([*varying, "detailed"]).s.value_counts().unstack("s", fill_value=0)

`show` displays what a function returned: a figure and the first rows of its data, a table's LaTeX, or a summary.
For `None` (not applicable) it shows nothing, and the warning above it says what's missing.

In [ ]:
def show(item, rows=5):
    """Display a `FigureResult` (figure, then the head of its data), print a table's LaTeX, or return a summary."""
    if item is None:
        return None
    if isinstance(item, FigureResult):
        display(item.figure)
        plt.close(item.figure)  # shown once; `item.figure` can still be displayed or saved
        return item.data.head(rows)
    if isinstance(item, str):
        print(item)
        return None
    return item

## Fig. 2 (and Fig. A1, A2): an example simulation

One detailed run: every agent's focal belief up to t = 100, with two example agents highlighted, and their BNs. The
defaults are the paper's: `seed=2`, `s=4`, `agents=(61, 79)`. `config="staticlow"` or `"statichigh"` gives Fig. A1 /
A2, which need detailed runs of those configs (`simOut_old_model` has none).

In [ ]:
fig2_result = fig2(SIM_DIR, **common)
show(fig2_result)

Nothing is saved until you ask. `save` writes `OUT_DIR/<name>.png` (600 dpi), plus a PDF if `"pdf"` is in
`formats`, creates `OUT_DIR` if needed and returns the paths.

In [ ]:
if fig2_result is not None:
    print(fig2_result.save(OUT_DIR, formats=FORMATS))

The numbers the paper quotes for this run: the example agents' balance (`tb_tot`, `tb_foc`) and personal dissonance
(`Hpers`) at t = 100, and how many agents hold a negative, zero or positive focal belief then.

In [ ]:
show(example_agent_stats(SIM_DIR, **common))

In [ ]:
show(focal_sign_counts(SIM_DIR, **common))

## Fig. 3 and Fig. A3: response frequencies

Fig. 3 compares the four BN configs (derived from `BASE`) over the pressure levels, with an example simulation in
panel A (a detailed adaptive run, `example_seed=1`, `example_s=4`). It uses every seed it finds: the old script
used 20, so pass `seeds=range(20)` to reproduce it. The polarisation summary gives σ_foc, mean |ω| and the other
polarisation numbers per config. Fig. A3 shows the detailed adaptive run `seed=1` at each of the six pressures, with
one random agent per response type highlighted (`rng_seed=14` picks the paper's).

In [ ]:
show(fig3(SIM_DIR, **common))

In [ ]:
show(polarisation_summary(SIM_DIR, **common))

In [ ]:
show(figA3(SIM_DIR, **common))

## Fig. 4, Fig. 5, Fig. A4 and Tables 3, A1–A7: BN metrics

These take a social-adaptation rate `mu`, which replaces the base's. μ = 0 gives the main text. μ = 0.005 and 0.05
give Fig. A5 / A6 and their tables (App. D), and they use only s = 4. Fig. 4 shows how the BN metrics tell the
response types apart (Cohen's d and logit coefficients per pressure), Fig. 5 how the metrics change over the three
windows, Fig. A4 their correlations, and `greedy_uncorrelated_subset` backs the choice of Fig. 4's predictors.

In [ ]:
MU = 0.0
show(fig4(SIM_DIR, mu=MU, **common))

In [ ]:
show(fig5(SIM_DIR, mu=MU, **common))

In [ ]:
show(figA4(SIM_DIR, mu=MU, **common))

In [ ]:
show(greedy_uncorrelated_subset(SIM_DIR, mu=MU, **common))

The tables are LaTeX strings with a file stem in `.name`. Table 3 is one table per pressure (`s=4` is Table 3, the
other pressures Tables A1–A4). `save_table` writes one to `OUT_DIR/tables/<name>.tex`.

In [ ]:
table = table3(SIM_DIR, s=4, mu=MU, **common)
show(table)

In [ ]:
if table is not None:
    print(save_table(table, OUT_DIR))

In [ ]:
show(tableA5(SIM_DIR, mu=MU, **common))

In [ ]:
show(tableA6(SIM_DIR, mu=MU, **common))

In [ ]:
show(tableA7(SIM_DIR, mu=MU, **common))

## Fig. 6 and Fig. A7: social adaptation

Fig. 6 runs over the social-adaptation rates μ that exist at s = 4 (the paper's eleven; the missing ones are warned
about), with the BNs of the example simulation `example_seed=1` at `example_mus=(0.005, 0.2)` in panels A and B.
Fig. A7 shows the distribution of each edge weight at t = 100 in one simulation (`seed=99`, `s=4`), one row per μ
in `mus=(0, 0.005, 0.2)`.

In [ ]:
show(fig6(SIM_DIR, **common))

In [ ]:
show(figA7(SIM_DIR, **common))

## Fig. A8: sensitivity

One parameter at a time around the base (M, N, β, p, τ for the three configs, ε and ω₀ for the adaptive one), at
s = 4. A row with a variant missing is left out, with a `MissingLevelsWarning`.

In [ ]:
show(figA8(SIM_DIR, **common))

## Everything at once: `make_all`

`make_all` runs every figure and table for `BASE`, saves them in `FORMATS` into `OUT_DIR` (tables into
`OUT_DIR/tables/`), skips the ones that don't apply and prints a report: the files written, and for each skipped
item why. It returns that report as a dict. With the shared loader and index, the CSVs read above aren't read
again. `figs=["fig4", "table3"]` restricts it, and `mus=` sets the μ values of the Fig. 4 family (default
0, 0.005, 0.05). From a shell: `python -m anb.analysis SIM_DIR OUT_DIR`.

In [ ]:
report = make_all(SIM_DIR, OUT_DIR, base=BASE, formats=FORMATS, loader=loader, index=index)